# **TP3** 
# UC06 - Exploration des données 

## Import des bibliothèques et création de la session Spark 

In [4]:
import os
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TP3-Retail-Intelligence")
    .master("local[*]")
    .getOrCreate()
)
print("Version Spark : ", spark.version)
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 17:26:54 WARN Utils: Your hostname, pauline-XMG-APEX-L23, resolves to a loopback address: 127.0.1.1; using 192.168.1.143 instead (on interface wlp4s0)
26/11/18 17:26:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 17:26:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/18 17:26:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Version Spark :  4.3.0


### Vérifications de l'environnement technique 

In [5]:
import os, sys, platform, pyspark

print("=== 1. Environnement Python ===")
print("Python        :", sys.version.split()[0])
print("Exécutable    :", sys.executable)
print("Env virtuel   :", os.environ.get("VIRTUAL_ENV"))
print("Système       :", platform.platform())

print("\n=== 2. Version PySpark ===")
print("PySpark       :", pyspark.__version__)

print("\n=== 3. Session Spark ===")
print("Session       :", spark)
print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)
print("App name      :", spark.sparkContext.appName)
print("Application ID:", spark.sparkContext.applicationId)
spark.createDataFrame([("Magasin A", 1200.0)], ["magasin", "chiffre_affaires"]).show()

print("\n=== 4. Interface de travail ===")
print("Jupyter dans VS Code (notebook :", "Notebook_TP3.ipynb", ")")

print("\n=== 5. Chemin des données ===")
path = os.path.expanduser(
    "~/Documents/IPSSI/DataLake Warehouse Lakehouse/TP/J3/data/raw/data/"
)
print("Chemin        :", path)
print("Existe        :", os.path.exists(path))

=== 1. Environnement Python ===
Python        : 3.14.4
Exécutable    : /home/pauline/project_retail-intelligence/.venv/bin/python
Env virtuel   : /home/pauline/project_retail-intelligence/.venv
Système       : Linux-7.0.0-43-generic-x86_64-with-glibc2.43

=== 2. Version PySpark ===
PySpark       : 4.3.0

=== 3. Session Spark ===
Session       : <pyspark.sql.session.SparkSession object at 0x7b967b545d30>
Spark version : 4.3.0
Master        : local[*]
App name      : TP3-Retail-Intelligence
Application ID: local-1795019215918


+---------+----------------+
|  magasin|chiffre_affaires|
+---------+----------------+
|Magasin A|          1200.0|
+---------+----------------+


=== 4. Interface de travail ===
Jupyter dans VS Code (notebook : Notebook_TP3.ipynb )

=== 5. Chemin des données ===
Chemin        : /home/pauline/Documents/IPSSI/DataLake Warehouse Lakehouse/TP/J3/data/raw/data/
Existe        : True


## Fichier store_sales.dat

### Chargement des données

In [8]:
path_ventes = os.path.expanduser(
    "~/Documents/IPSSI/DataLake Warehouse Lakehouse/TP/J3/data/raw/data/store_sales/store_sales.dat"
)

df_sales = (
    spark.read
    .option("header", "False")
    .option("inferSchema", "True")
    .option("delimiter", "|")
    .csv("file://" + path_ventes)
)

print("Nombre de lignes table sales:", df_sales.count () )

Nombre de lignes table sales: 667579


### Affichage du schéma

In [9]:
df_sales.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: integer (nullable = true)
 |-- _c2: integer (nullable = true)
 |-- _c3: integer (nullable = true)
 |-- _c4: integer (nullable = true)
 |-- _c5: integer (nullable = true)
 |-- _c6: integer (nullable = true)
 |-- _c7: integer (nullable = true)
 |-- _c8: integer (nullable = true)
 |-- _c9: integer (nullable = true)
 |-- _c10: integer (nullable = true)
 |-- _c11: double (nullable = true)
 |-- _c12: double (nullable = true)
 |-- _c13: double (nullable = true)
 |-- _c14: double (nullable = true)
 |-- _c15: double (nullable = true)
 |-- _c16: double (nullable = true)
 |-- _c17: double (nullable = true)
 |-- _c18: double (nullable = true)
 |-- _c19: double (nullable = true)
 |-- _c20: double (nullable = true)
 |-- _c21: double (nullable = true)
 |-- _c22: double (nullable = true)



### Echantillon de lignes

In [10]:
df_sales.show(5, truncate=False)

+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-------+-------+------+------+-------+-------+-------+
|_c0  |_c1  |_c2  |_c3  |_c4    |_c5 |_c6  |_c7|_c8|_c9|_c10|_c11 |_c12  |_c13 |_c14   |_c15   |_c16   |_c17   |_c18  |_c19  |_c20   |_c21   |_c22   |
+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-------+-------+------+------+-------+-------+-------+
|36890|71480|10001|44404|493944 |2898|44093|10 |294|0  |31  |18.0 |30.78 |23.08|238.7  |715.48 |558.0  |954.18 |21.46 |0.0   |715.48 |736.94 |157.48 |
|36890|71480|10000|44404|493944 |2898|44093|10 |279|0  |54  |99.21|148.82|83.34|3535.92|4500.36|5357.34|8036.28|225.02|0.0   |4500.36|4725.38|-856.98|
|36890|71480|15556|44404|493944 |2898|44093|10 |114|0  |39  |79.49|94.59 |83.24|442.65 |3246.36|3100.11|3689.01|184.07|616.81|2629.55|2813.62|-470.56|
|36890|45392|4597 |89832|1774847|6266|16285|4  |285|1  |20  |74.38|136.12|133.4|54.4   |2668.0

### Compter les enregistrements

In [11]:
print("Nombre de lignes :", df_sales.count())
print("Nombre de colonnes :", len(df_sales.columns))

Nombre de lignes : 667579
Nombre de colonnes : 23


### Examiner les colonnes

In [12]:
for nom_colonne, type_colonne in df_sales.dtypes:
    print(nom_colonne, ":", type_colonne)

_c0 : int
_c1 : int
_c2 : int
_c3 : int
_c4 : int
_c5 : int
_c6 : int
_c7 : int
_c8 : int
_c9 : int
_c10 : int
_c11 : double
_c12 : double
_c13 : double
_c14 : double
_c15 : double
_c16 : double
_c17 : double
_c18 : double
_c19 : double
_c20 : double
_c21 : double
_c22 : double


### Identifier les valeurs manquantes 

In [13]:
from pyspark.sql import functions as F

for nom_colonne in df_sales.columns:
    nb_vides = df_sales.filter(
        F.col(nom_colonne).isNull() 
    ).count()
    print(nom_colonne, ":", nb_vides, "valeur.s nulle.s"
)

_c0 : 0 valeur.s nulle.s
_c1 : 0 valeur.s nulle.s
_c2 : 0 valeur.s nulle.s
_c3 : 0 valeur.s nulle.s
_c4 : 238 valeur.s nulle.s
_c5 : 295 valeur.s nulle.s
_c6 : 448 valeur.s nulle.s
_c7 : 0 valeur.s nulle.s
_c8 : 158 valeur.s nulle.s
_c9 : 0 valeur.s nulle.s
_c10 : 0 valeur.s nulle.s
_c11 : 0 valeur.s nulle.s
_c12 : 0 valeur.s nulle.s
_c13 : 0 valeur.s nulle.s
_c14 : 0 valeur.s nulle.s
_c15 : 0 valeur.s nulle.s
_c16 : 0 valeur.s nulle.s
_c17 : 0 valeur.s nulle.s
_c18 : 0 valeur.s nulle.s
_c19 : 0 valeur.s nulle.s
_c20 : 0 valeur.s nulle.s
_c21 : 0 valeur.s nulle.s
_c22 : 0 valeur.s nulle.s


### Rechercher les doublons (si existants)

In [14]:
nb_lignes = df_sales.count()
nb_sans_doublon = df_sales.dropDuplicates().count()
print("Lignes entièrement identiques :", nb_lignes - nb_sans_doublon)

Lignes entièrement identiques : 0


### Repérer les valeurs atypiques

In [15]:
for nom_colonne, type_colonne in df_sales.dtypes:
    if type_colonne in ("int", "double"):
        stats = df_sales.agg(
            F.min(nom_colonne).alias("min"),
            F.max(nom_colonne).alias("max"),
            F.avg(nom_colonne).alias("moyenne"),
        ).first()
        print(nom_colonne, ": min =", stats["min"], "| max =", stats["max"], "| moyenne =", round(stats["moyenne"], 2))

_c0 : min = 36890 | max = 38697 | moyenne = 37792.24
_c1 : min = 0 | max = 86399 | moyenne = 43304.89
_c2 : min = 0 | max = 17819 | moyenne = 9197.34
_c3 : min = 2 | max = 98999 | moyenne = 50415.29
_c4 : min = 18 | max = 1920742 | moyenne = 955241.37
_c5 : min = 0 | max = 7199 | moyenne = 3615.64
_c6 : min = 0 | max = 49499 | moyenne = 24730.22
_c7 : min = 0 | max = 11 | moyenne = 5.5
_c8 : min = 0 | max = 299 | moyenne = 149.74
_c9 : min = 0 | max = 89099 | moyenne = 44494.56
_c10 : min = 1 | max = 100 | moyenne = 50.56
_c11 : min = 1.0 | max = 100.0 | moyenne = 50.49
_c12 : min = 1.0 | max = 200.0 | moyenne = 75.73
_c13 : min = 0.0 | max = 199.3 | moyenne = 37.86
_c14 : min = 0.0 | max = 18984.0 | moyenne = 1912.45
_c15 : min = 0.0 | max = 18745.0 | moyenne = 1913.22
_c16 : min = 1.0 | max = 9997.0 | moyenne = 2551.13
_c17 : min = 1.2 | max = 19814.0 | moyenne = 3825.67
_c18 : min = 0.0 | max = 1637.8 | moyenne = 77.43
_c19 : min = 0.0 | max = 14371.66 | moyenne = 189.78
_c20 : min 

## Fichier item.dat

### Chargement des données

In [19]:
path_item = os.path.expanduser(
    "~/Documents/IPSSI/DataLake Warehouse Lakehouse/TP/J3/data/raw/data/item/item.dat"
)

df_item = (
    spark.read
    .option("header", "False")
    .option("inferSchema", "True")
    .option("delimiter", "|")
    .csv("file://" + path_item)
)

print("Nombre de lignes table item:", df_item.count () )

Nombre de lignes table item: 17820


### Affichage du schéma

In [ ]:
df_item.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: date (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: double (nullable = true)
 |-- _c6: double (nullable = true)
 |-- _c7: integer (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: integer (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: integer (nullable = true)
 |-- _c12: string (nullable = true)
 |-- _c13: integer (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)
 |-- _c16: string (nullable = true)
 |-- _c17: string (nullable = true)
 |-- _c18: string (nullable = true)
 |-- _c19: string (nullable = true)
 |-- _c20: integer (nullable = true)
 |-- _c21: string (nullable = true)



### Echantillons de lignes

In [ ]:
df_item.show(5, truncate=False)

+---+----------------+----------+----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+-----+-------+---------------------------------+---+---------------------+----+-----------------+----+--------------------------------------------------+------+--------------------+--------+-----+-------+----+------------------------------------------------+
|_c0|_c1             |_c2       |_c3 |_c4                                                                                                                                                                                                |_c5  |_c6  |_c7    |_c8                              |_c9|_c10                 |_c11|_c12             |_c13|_c14                                              |_c15  |_c16                |_c17    |_c18 |_c19   |_c20|_c21                                            |
+-

### Compter les enregistrements

In [ ]:
print("Nombre de lignes :", df_item.count())
print("Nombre de colonnes :", len(df_item.columns))

Nombre de lignes : 17820
Nombre de colonnes : 22


### Examiner les colonnes

In [ ]:
for nom_colonne, type_colonne in df_item.dtypes:
    print(nom_colonne, ":", type_colonne)

_c0 : int
_c1 : string
_c2 : date
_c3 : string
_c4 : string
_c5 : double
_c6 : double
_c7 : int
_c8 : string
_c9 : int
_c10 : string
_c11 : int
_c12 : string
_c13 : int
_c14 : string
_c15 : string
_c16 : string
_c17 : string
_c18 : string
_c19 : string
_c20 : int
_c21 : string


### Identifier les valeurs manquantes 

In [ ]:
from pyspark.sql import functions as F

for nom_colonne in df_item.columns:
    nb_vides = df_item.filter(
        F.col(nom_colonne).isNull() 
    ).count()
    print(nom_colonne, ":", nb_vides, "valeur.s nulle.s"
)

_c0 : 0 valeur.s nulle.s
_c1 : 0 valeur.s nulle.s
_c2 : 0 valeur.s nulle.s
_c3 : 17820 valeur.s nulle.s
_c4 : 4 valeur.s nulle.s
_c5 : 0 valeur.s nulle.s
_c6 : 0 valeur.s nulle.s
_c7 : 8 valeur.s nulle.s
_c8 : 6 valeur.s nulle.s
_c9 : 0 valeur.s nulle.s
_c10 : 0 valeur.s nulle.s
_c11 : 0 valeur.s nulle.s
_c12 : 0 valeur.s nulle.s
_c13 : 0 valeur.s nulle.s
_c14 : 0 valeur.s nulle.s
_c15 : 0 valeur.s nulle.s
_c16 : 0 valeur.s nulle.s
_c17 : 0 valeur.s nulle.s
_c18 : 0 valeur.s nulle.s
_c19 : 0 valeur.s nulle.s
_c20 : 0 valeur.s nulle.s
_c21 : 0 valeur.s nulle.s


### Recherhcer les doublons (si existants)

In [ ]:
nb_lignes = df_item.count()
nb_sans_doublon = df_item.dropDuplicates().count()
print("Lignes entièrement identiques :", nb_lignes - nb_sans_doublon)

Lignes entièrement identiques : 0


### Repérer les valeurs atypiques 

In [ ]:
for nom_colonne, type_colonne in df_item.dtypes:
    if type_colonne in ("int", "double"):
        stats = df_item.agg(
            F.min(nom_colonne).alias("min"),
            F.max(nom_colonne).alias("max"),
            F.avg(nom_colonne).alias("moyenne"),
        ).first()
        print(nom_colonne, ": min =", stats["min"], "| max =", stats["max"], "| moyenne =", round(stats["moyenne"], 2))

_c0 : min = 0 | max = 17819 | moyenne = 8909.5
_c5 : min = 0.1 | max = 99.98 | moyenne = 50.17
_c6 : min = 0.02 | max = 87.35 | moyenne = 43.66
_c7 : min = 1001703 | max = 10015586 | moyenne = 5520673.64
_c9 : min = 1 | max = 16 | moyenne = 4.94
_c11 : min = 1 | max = 9 | moyenne = 3.28
_c13 : min = 1 | max = 1000 | moyenne = 501.69
_c20 : min = 1 | max = 100 | moyenne = 50.63


## Tests supplémentaires pour la partie nettoyage

In [16]:
print("Ventes à prix 0 (_c13 = 0) :", df_sales.filter(F.col("_c13") == 0).count())

Ventes à prix 0 (_c13 = 0) : 3324


In [20]:
print("item, _c6 > _c5 :", df_item.filter(F.col("_c6") > F.col("_c5")).count())

item, _c6 > _c5 : 7768


In [21]:
df_item.agg(F.min("_c2"), F.max("_c2")).show()

+----------+----------+
|  min(_c2)|  max(_c2)|
+----------+----------+
|2000-01-03|2000-12-31|
+----------+----------+



## Exploration du date_dim pour conversion des cols en date 

In [23]:
path_date = os.path.expanduser(
    "~/Documents/IPSSI/DataLake Warehouse Lakehouse/TP/J3/data/raw/data/date_dim/date_dim.dat"
)

df_date = (
    spark.read
    .option("header", "False")
    .option("inferSchema", "True")
    .option("delimiter", "|")
    .csv("file://" + path_date)
)
df_date.show(5, truncate=False)

# Toutes les clés de date des ventes existent-elles dans date_dim ?
absentes = (df_sales.select("_c0").distinct()
            .join(df_date.select(F.col("_c0").alias("id_date")),
                  F.col("_c0") == F.col("id_date"), "left_anti")
            .count())
print("Clés de date des ventes absentes de date_dim :", absentes)

26/11/18 17:37:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---+----------------+----------+---+---+---+----+---+---+---+----+----+----+----+---------+------+----+----+----+-------+-------+-------+-------+----+----+----+----+----+
|_c0|_c1             |_c2       |_c3|_c4|_c5|_c6 |_c7|_c8|_c9|_c10|_c11|_c12|_c13|_c14     |_c15  |_c16|_c17|_c18|_c19   |_c20   |_c21   |_c22   |_c23|_c24|_c25|_c26|_c27|
+---+----------------+----------+---+---+---+----+---+---+---+----+----+----+----+---------+------+----+----+----+-------+-------+-------+-------+----+----+----+----+----+
|0  |AAAAAAAAAAAAAAAA|1900-01-01|0  |0  |0  |1900|1  |1  |1  |1   |1900|0   |0   |Monday   |1900Q1|Y   |N   |N   |2448812|2458802|2472542|2420941|N   |N   |N   |N   |N   |
|1  |AAAAAAAAAAAAAAAB|1900-01-02|0  |0  |0  |1900|2  |1  |2  |1   |1900|0   |0   |Tuesday  |1900Q1|Y   |N   |N   |2441612|2427135|2449909|2430434|N   |N   |N   |N   |N   |
|2  |AAAAAAAAAAAAAAAC|1900-01-03|0  |0  |0  |1900|3  |1  |3  |1   |1900|0   |0   |Wednesday|1900Q1|Y   |N   |N   |2436156|2449280|2455660|24